# Day 4 — Evaluate BaselineCNN on the held-out test set

Run this on **Kaggle** with:
- the `emmarex/plantdisease` dataset added (Add Input -> search `plantdisease`),
- the Day 3 checkpoint available: either run it in the **same session** as Day 3 (so
  `/kaggle/working/outputs/baseline_best.pt` still exists), or upload `baseline_best.pt` **and**
  `metrics_baseline.json` together as a private Kaggle Dataset and add it as an input,
- **Accelerator: GPU** if you can (CPU works too for a single pass over the test set, just slower),
- **Internet on**, so the cell below can clone the `leafguard` repo.

Goal: score the Day 3 model **once** on the seeded 70/15/15 test split it never saw during training or checkpoint
selection. `evaluate()` rebuilds the split with the same seed and, when `metrics_baseline.json` is available,
checks the rebuilt test indices against the ones recorded at training time.

Outputs land in `/kaggle/working/outputs/`: `eval_baseline.json`, `confusion_matrix_baseline.png`, `baseline_eval.md`.

In [ ]:
import os

# Kaggle has used both mount layouts; Day 1 found the second one.
CANDIDATE_ROOTS = [
    "/kaggle/input/plantdisease/PlantVillage",
    "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
assert DATA_ROOT, f"PlantVillage not found in any of {CANDIDATE_ROOTS} -- is the dataset added?"
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
import shutil
import subprocess
import sys

# Use an uploaded copy of src/ if there is one; otherwise clone the repo (needs Internet on).
# The Day 4 code must be on this ref: if the Day 4 PR isn't merged yet, use "day-4-baseline-eval".
LEAFGUARD_REF = "main"
CLONE_DIR = "/kaggle/working/leafguard"

for path in ["/kaggle/input/leafguard-src", f"{CLONE_DIR}/src"]:
    if os.path.isdir(path) and path not in sys.path:
        sys.path.insert(0, path)

try:
    import leafguard.evaluate  # noqa: F401
except ImportError:
    # A Day 3 clone in the same session has no evaluate.py: replace it with a fresh one.
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", LEAFGUARD_REF,
         "https://github.com/Bloop31/leafguard.git", CLONE_DIR],
        check=True,
    )
    sys.path.insert(0, f"{CLONE_DIR}/src")
    for mod in [m for m in sys.modules if m == "leafguard" or m.startswith("leafguard.")]:
        del sys.modules[mod]  # drop a stale pre-Day-4 copy imported above
    import leafguard.evaluate  # noqa: F401

import leafguard
from leafguard.evaluate import evaluate

print("leafguard", leafguard.__version__, "from", leafguard.__file__)

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- evaluating on CPU. That's fine for one pass over the test set, just slower.")

In [ ]:
import glob

OUT_DIR = "/kaggle/working/outputs"
BACKBONE = "baseline"


def find_one(name):
    """Same-session Day 3 output first, then any uploaded Kaggle Dataset."""
    hits = [p for p in [f"{OUT_DIR}/{name}"] if os.path.isfile(p)]
    hits += sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None


WEIGHTS_PATH = find_one(f"{BACKBONE}_best.pt")
TRAIN_METRICS_PATH = find_one(f"metrics_{BACKBONE}.json")
assert WEIGHTS_PATH, (
    f"{BACKBONE}_best.pt not found in {OUT_DIR} or /kaggle/input -- upload the Day 3 checkpoint as a dataset."
)
print("Weights:         ", WEIGHTS_PATH)
print("Training metrics:", TRAIN_METRICS_PATH or "not found -- the split cross-check will be skipped")

In [ ]:
import time

t0 = time.time()
metrics = evaluate(
    backbone=BACKBONE,
    weights_path=WEIGHTS_PATH,
    data_root=DATA_ROOT,
    out_dir=OUT_DIR,
    batch_size=32,
    seed=42,
    metrics_path=TRAIN_METRICS_PATH,
)
print(f"Evaluation time: {time.time() - t0:.0f}s")

In [ ]:
from IPython.display import Image, Markdown, display

display(Image(filename=f"{OUT_DIR}/confusion_matrix_{BACKBONE}.png"))

In [ ]:
with open(f"{OUT_DIR}/{BACKBONE}_eval.md") as f:
    display(Markdown(f.read()))

In [ ]:
from leafguard.evaluate import worst_classes

print(f"Test accuracy: {metrics['accuracy']:.4f}  (n={metrics['n_test']}, split check: {metrics['split']['check']})")
print(f"Macro F1:      {metrics['macro_f1']:.4f}")
print("Worst 3 classes by F1:")
for cls, f1 in worst_classes(metrics, n=3):
    print(f"  {cls:<45} {f1:.4f}")
print("Most-confused pairs (true -> predicted):")
for c in metrics["top_confusions"]:
    print(f"  {c['true']} -> {c['predicted']}: {c['count']} ({c['rate']:.1%} of {c['true']})")
print()
print("Files to download from", OUT_DIR, "(.json/.md -> reports/, .png -> reports/figures/)")
for name in sorted(os.listdir(OUT_DIR)):
    if name.startswith(("eval_", "confusion_matrix_")) or name.endswith("_eval.md"):
        print(f"  {name:<32} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1e6:.2f} MB")

## Findings (fill this in after running)

- Test accuracy vs Day 3 best val accuracy (big drop = the val-based checkpoint choice was lucky):
- Macro F1 vs accuracy (a large gap means small classes are doing worse):
- Worst 3 classes by F1:
- Most-confused pairs (same crop, different disease? or across crops?):
- Anything unexpected?